# **Data Cleaning Pipeline for Multiple Tabs or Sheets**

Luis Felipe Villota

InSight Crime - MAD Unit 

---------------------


In [1]:
import time

start_time = time.time()
formatted_start_time = time.ctime(start_time)
print(f"Start time: {formatted_start_time}")

Start time: Mon May  4 10:45:28 2026


## Setup

#### Version Control

The project is created within a single GitHub repository. I keep the repository `private` with the possibility to give access to the online repo at any time. 

#### Environment

In [3]:
import os
import sys
import shutil
import subprocess

venv_name = "venv-cuina"
venv_path = os.path.join(os.getcwd(), venv_name)

# ── Portable venv Python resolver ────────────────────────────────────────────
def get_venv_python(venv_path):
    """Find the Python executable regardless of OS or how the venv was created."""
    candidates = [
        os.path.join(venv_path, "bin",     "python"),        # Linux / macOS
        os.path.join(venv_path, "bin",     "python3"),       # some Linux builds
        os.path.join(venv_path, "Scripts", "python.exe"),    # Windows
        os.path.join(venv_path, "Scripts", "python"),        # Windows (no ext)
    ]
    return next((p for p in candidates if os.path.isfile(p)), None)

# ── Check if existing venv is usable ────────────────────────────────────────
if os.path.exists(venv_path):
    python = get_venv_python(venv_path)
    if python:
        print(f"✅ Venv '{venv_name}' exists and is healthy.")
    else:
        print(f"⚠️  Venv exists but has no usable Python — rebuilding...")
        shutil.rmtree(venv_path)

# ── Create if missing (or was just wiped) ────────────────────────────────────
if not os.path.exists(venv_path):
    try:
        subprocess.run([sys.executable, "-m", "venv", venv_path], check=True)
        print(f"✅ Created venv: {venv_name}")
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"❌ Failed to create venv: {e}")

# ── Re-resolve after possible rebuild ────────────────────────────────────────
python = get_venv_python(venv_path)
if not python:
    raise FileNotFoundError(
        f"Still no Python found in {venv_path}\n"
        f"Contents: {os.listdir(venv_path)}"
    )

print(f"🐍 Python: {python}")

# ── Install packages ──────────────────────────────────────────────────────────
packages = [
    "gspread", "pandas", "requests", "google-api-python-client",
    "oauth2client", "matplotlib", "seaborn", "missingno",
    "gspread-formatting", "ipykernel", "openpyxl",
    "gspread-dataframe", "pytz", "feedparser"
]

def normalize(name):
    return name.lower().replace("-", "_")

try:
    result = subprocess.run(
        [python, "-m", "pip", "freeze"],
        capture_output=True, text=True, check=True
    )
    installed = {
        normalize(line.split("==")[0])
        for line in result.stdout.splitlines() if "==" in line
    }
except subprocess.CalledProcessError:
    installed = set()

missing = [pkg for pkg in packages if normalize(pkg) not in installed]

if missing:
    print(f"⚠️  Installing: {missing}")
    subprocess.run([python, "-m", "pip", "install", "--upgrade", "pip"], check=True)
    subprocess.run([python, "-m", "pip", "install"] + missing, check=True)
    print("✅ Packages installed.")
else:
    print("✅ All packages already present.")

# ── Register Jupyter kernel ───────────────────────────────────────────────────
try:
    subprocess.run([
        python, "-m", "ipykernel", "install",
        "--user", "--name", "cuina-kernel", "--display-name", "Python (cuina)"
    ], check=True)
    print("✅ Registered kernel: 'Python (cuina)'")
except subprocess.CalledProcessError as e:
    raise RuntimeError(f"❌ Kernel registration failed: {e}")

# ── Freeze requirements ───────────────────────────────────────────────────────
req_path = os.path.join(os.getcwd(), "requirements.txt")
with open(req_path, "w") as f:
    subprocess.run([python, "-m", "pip", "freeze"], stdout=f, check=True)

print(f"✅ requirements.txt → {req_path}")

✅ Created venv: venv-cuina
🐍 Python: /home/lfvm/codenivore/codebaker/garde-manger/ic-audiences/venv-cuina/bin/python
⚠️  Installing: ['gspread', 'pandas', 'requests', 'google-api-python-client', 'oauth2client', 'matplotlib', 'seaborn', 'missingno', 'gspread-formatting', 'ipykernel', 'openpyxl', 'gspread-dataframe', 'pytz', 'feedparser']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 17.1 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1
  Using cached gspread-6.2.1-py3-none-any.whl.metadata (11 kB)
  Using cached pandas-3.0.2-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached oauth2client-4.1.3-py2.py3-none-any.whl.metadata (1.2 kB)
  Using cached matplotlib-3.10.9-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached seaborn-0.13.

In [4]:
import os
from datetime import datetime

# Get current date
print("Current Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

# Count items, files, and directories
items = os.listdir()
files = [f for f in items if os.path.isfile(f)]
directories = [d for d in items if os.path.isdir(d)]

print(f"Total Items: {len(items)}")
print(f"Files: {len(files)}")
print(f"Directories: {len(directories)}")

# Get files sorted by modification time 
print("\nJust files (newest first):")
files_with_mtime = [(f, os.path.getmtime(f)) for f in items if os.path.isfile(f)]
sorted_files = sorted(files_with_mtime, key=lambda x: x[1], reverse=True)

for file, mtime in sorted_files:
    print(f"{datetime.fromtimestamp(mtime)} - {file}")

Current Date: 2026-05-04 09:56:31
Total Items: 9
Files: 5
Directories: 4

Just files (newest first):
2026-05-04 09:56:23.476590 - initial_config.ipynb
2026-05-04 09:56:20.065791 - requirements.txt
2026-04-07 07:59:45.016569 - by_sector_clean.xlsx
2026-04-07 07:29:38.305458 - mailchimp-clean-upload.csv
2026-04-03 14:54:37.332517 - .gitignore


#### Libraries

In [5]:
import re
import requests
import pandas as pd
from datetime import datetime
from google.oauth2 import service_account
from googleapiclient.discovery import build
import gspread
from google.oauth2.service_account import Credentials
from gspread_formatting import format_cell_ranges, CellFormat, Color
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import openpyxl as px
import gspread_dataframe as gd

### API

This creates a modular client (frontend) call that is able to extract the desired subset of data from the API server (backend); -and, make it easily reusable for future queries.


In [6]:
# Path to your service account key file
SERVICE_ACCOUNT_FILE = '/home/lfvm/codenivore/codebaker/nyckelring/brelok/summer-sector-439022-v6-2eafffbbfb90.json'

# Update with the actual path or team credentials file

# r'C:\Users\lfvm0\Desktop\summer-sector-439022-v6-0988365c484e.json'

# '/home/lfvm/codenivore/codebaker/nyckelring/brelok/summer-sector-439022-v6-2eafffbbfb90.json'

# VERY IMPORTANT: Ensure the service account has access to the Google Sheet by sharing it with the service account email

# Original Google Sheet ID (latest version) 
ORIGINAL_SPREADSHEET_ID =  '1x9q9MJcHynAjr14CdvygFFDUWoHRM1Y0qtoIMrgkwHg'

# Define scopes for Google Sheets and Drive API
SCOPES = ['https://www.googleapis.com/auth/spreadsheets', 
          'https://www.googleapis.com/auth/drive']

GOOGLE_API_KEY = '2eafffbbfb905d4cef6a81b4bff19cfabf7adf0b'
GOOGLE_CX = '65163f283105448fa'

In [7]:
# Authenticate and build both Sheets and Drive services
creds = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES)
sheet_service = build('sheets', 'v4', credentials=creds)
drive_service = build('drive', 'v3', credentials=creds)

#### First call

In [8]:
# Original spreadsheet
# Last modification: user and time. This information is related to the file itself, not to specific content (sheets, cells, rows)

from googleapiclient.discovery import build
from datetime import datetime
import pytz

def get_last_modifying_user(drive_service, file_id):
    try:
        # Get revisions (only the last one)
        revisions = drive_service.revisions().list(
            fileId=file_id,
            fields="revisions(modifiedTime,lastModifyingUser)"
        ).execute().get('revisions', [])
        
        if not revisions:
            return "No revisions found."
        
        last_revision = revisions[-1]  # Most recent revision
        user_info = last_revision.get('lastModifyingUser', {})
        email = user_info.get('emailAddress', 'Unknown')
        modified_time = last_revision.get('modifiedTime', 'Unknown')
        
        # Convert to readable datetime
        if modified_time != 'Unknown':
            dt = datetime.strptime(modified_time, "%Y-%m-%dT%H:%M:%S.%fZ")
            dt = dt.replace(tzinfo=pytz.UTC)
            modified_time = dt.strftime("%Y-%m-%d %H:%M:%S (UTC)")
        
        return {
            "user_email": email,
            "modified_time": modified_time
        }
    
    except Exception as e:
        return f"Error: {str(e)}"

# Usage
# First get the file name and sheet name
file_metadata = drive_service.files().get(fileId=ORIGINAL_SPREADSHEET_ID, fields='name').execute()
file_name = file_metadata.get('name', 'Unknown')

# Get sheet names
sheets = sheet_service.spreadsheets().get(
    spreadsheetId=ORIGINAL_SPREADSHEET_ID,
    fields='sheets(properties(title))'
).execute().get('sheets', [])

sheet_names = [sheet['properties']['title'] for sheet in sheets]

# Get last modifier info
last_modifier = get_last_modifying_user(drive_service, ORIGINAL_SPREADSHEET_ID)


print(
    f"📄 File name: {file_name}\n"
    f"📑 Sheet names: {', '.join(sheet_names)}\n"
    f"🔄 Last modified by: {last_modifier['user_email']}\n"
    f"⏰ Last modified at: {last_modifier['modified_time']}"
)

📄 File name: mailchimp-clean-upload
📑 Sheet names: ONGS, Argentina , Homicidios, Drogas Sinteticas, Pandillas, Tennesse media, Migration (US civil society and instutional), Ambiente, Narcotrafico, Esequibo,  Venezuela, Elites y Crimen, Género, Paz Colombia, US terrorism event - Civil, US terrorism event - Govt
🔄 Last modified by: lvillota@insightcrime.org
⏰ Last modified at: 2026-04-05 02:09:18 (UTC)


## Extraction


### Tab names

In [ ]:
spreadsheet_metadata = sheet_service.spreadsheets().get(
    spreadsheetId=ORIGINAL_SPREADSHEET_ID
).execute()

sheets = spreadsheet_metadata.get('sheets', [])
sheet_names = [s['properties']['title'] for s in sheets]

print(sheet_names)

['ONGS', 'Argentina ', 'Homicidios', 'Drogas Sinteticas', 'Pandillas', 'Tennesse media', 'Migration (US civil society and instutional)', 'Ambiente', 'Narcotrafico', 'Esequibo', ' Venezuela', 'Elites y Crimen', 'Género', 'Paz Colombia', 'US terrorism event - Civil', 'US terrorism event - Govt']


### Import and summarize

In [ ]:
import pandas as pd

sheets_dict = {}

for sheet_name in sheet_names:
    result = sheet_service.spreadsheets().values().get(
        spreadsheetId=ORIGINAL_SPREADSHEET_ID,
        range=sheet_name
    ).execute()
    
    values = result.get('values', [])
    
    if not values or len(values) < 2:
        print(f"[SKIP] {sheet_name}: insufficient data (rows: {len(values)})")
        continue
    
    headers = values[0]
    rows = values[1:]
    
    # --- FIX: normalize row lengths ---
    max_len = max(len(headers), max(len(r) for r in rows))
    
    # expand headers if needed
    headers = headers + [f"extra_col_{i}" for i in range(len(headers), max_len)]
    
    fixed_rows = []
    for i, row in enumerate(rows):
        if len(row) != len(headers):
            print(f"[WARNING] {sheet_name} row {i} has {len(row)} cols, expected {len(headers)}")
        
        row = row + [""] * (len(headers) - len(row))  # pad missing
        fixed_rows.append(row[:len(headers)])         # trim overflow
    
    df = pd.DataFrame(fixed_rows, columns=headers)
    
    # --- clean columns ---
    df.columns = df.columns.str.strip().str.lower()
    
    # --- add metadata ---
    df["source_sheet"] = sheet_name
    df["theme"] = sheet_name
    
    sheets_dict[sheet_name] = df

# --- PRODUCE DIMENSIONS AND COLUMNS TABLE ---
print("\n" + "="*80)
print("TABLE: Sheet Dimensions and Column Names")
print("="*80)

# Create a list to store the table data
table_data = []

for sheet_name, df in sheets_dict.items():
    rows, cols = df.shape
    column_list = ", ".join(df.columns.tolist())
    
    table_data.append({
        "Sheet Name": sheet_name,
        "Rows": rows,
        "Columns": cols,
        "Column Names": column_list
    })

# Create DataFrame for display
summary_df = pd.DataFrame(table_data)

# Display the table
print(summary_df.to_string(index=False))

# Optional: Save to CSV
# summary_df.to_csv("sheet_dimensions_and_columns.csv", index=False)
# print("\n[SAVED] Table exported to 'sheet_dimensions_and_columns.csv'")

# Optional: Display detailed view per sheet
print("\n" + "="*80)
print("DETAILED VIEW PER SHEET")
print("="*80)

for sheet_name, df in sheets_dict.items():
    print(f"\n📊 {sheet_name}")
    print(f"   Dimensions: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"   Columns ({len(df.columns)}):")
    for i, col in enumerate(df.columns, 1):
        print(f"      {i}. {col}")
    print("-"*50)

[WARNING] ONGS row 0 has 8 cols, expected 10
[WARNING] ONGS row 1 has 8 cols, expected 10
[WARNING] ONGS row 2 has 8 cols, expected 10
[WARNING] ONGS row 3 has 8 cols, expected 10
[WARNING] ONGS row 4 has 8 cols, expected 10
[WARNING] ONGS row 5 has 8 cols, expected 10
[WARNING] ONGS row 6 has 8 cols, expected 10
[WARNING] ONGS row 7 has 8 cols, expected 10
[WARNING] ONGS row 8 has 8 cols, expected 10
[WARNING] ONGS row 9 has 8 cols, expected 10
[WARNING] ONGS row 10 has 8 cols, expected 10
[WARNING] ONGS row 11 has 8 cols, expected 10
[WARNING] ONGS row 12 has 8 cols, expected 10
[WARNING] ONGS row 13 has 8 cols, expected 10
[WARNING] ONGS row 14 has 8 cols, expected 10
[WARNING] ONGS row 15 has 8 cols, expected 10
[WARNING] ONGS row 16 has 8 cols, expected 10
[WARNING] ONGS row 17 has 8 cols, expected 10
[WARNING] ONGS row 18 has 8 cols, expected 10
[WARNING] ONGS row 19 has 8 cols, expected 10
[WARNING] ONGS row 20 has 8 cols, expected 10
[WARNING] ONGS row 21 has 8 cols, expected 1

## Transformation

### Boundaries

In [ ]:
def trim_sheet_to_data(sheet_service, spreadsheet_id, sheet_name):
    """Remove blank rows and columns after the end of actual data in the sheet"""
    
    # Get sheet details without grid data
    spreadsheet = sheet_service.spreadsheets().get(
        spreadsheetId=spreadsheet_id,
        fields="sheets(properties(sheetId,title,gridProperties(rowCount,columnCount)))"
    ).execute()

    sheet_id = None
    total_row_count = 0
    total_col_count = 0
    
    for sheet in spreadsheet["sheets"]:
        if sheet["properties"]["title"] == sheet_name:
            sheet_id = sheet["properties"]["sheetId"]
            total_row_count = sheet["properties"]["gridProperties"]["rowCount"]
            total_col_count = sheet["properties"]["gridProperties"]["columnCount"]
            break

    if sheet_id is None:
        raise ValueError(f"Sheet '{sheet_name}' not found")

    print(f"\n{'='*60}")
    print(f"TRIMMING TAB: '{sheet_name}'")
    print(f"{'='*60}")
    print(f"Current dimensions: {total_row_count} rows × {total_col_count} columns")
    
    # Get all data at once for more efficient column scanning
    try:
        result = sheet_service.spreadsheets().values().get(
            spreadsheetId=spreadsheet_id,
            range=sheet_name,
            majorDimension="ROWS"
        ).execute()
        
        values = result.get('values', [])
        
        if not values:
            print(f"No data found in sheet '{sheet_name}'. No trimming needed.")
            return total_row_count, total_col_count
        
        # Find last row with data
        last_data_row = 0
        for row_idx in range(len(values) - 1, -1, -1):
            row = values[row_idx]
            has_data = any(cell and str(cell).strip() for cell in row)
            if has_data:
                last_data_row = row_idx + 1  # Convert to 1-based index
                break
        
        # Find last column with data
        last_data_col = 0
        # Transpose to scan by columns
        max_cols = max(len(row) for row in values) if values else 0
        for col_idx in range(max_cols - 1, -1, -1):
            has_data = False
            for row in values:
                if col_idx < len(row) and row[col_idx] and str(row[col_idx]).strip():
                    has_data = True
                    break
            if has_data:
                last_data_col = col_idx + 1  # Convert to 1-based index
                break
        
        # Prepare batch update requests
        requests = []
        
        # Delete blank rows if needed
        if last_data_row < total_row_count:
            requests.append({
                "deleteDimension": {
                    "range": {
                        "sheetId": sheet_id,
                        "dimension": "ROWS",
                        "startIndex": last_data_row,
                        "endIndex": total_row_count
                    }
                }
            })
        
        # Delete blank columns if needed
        if last_data_col < total_col_count:
            requests.append({
                "deleteDimension": {
                    "range": {
                        "sheetId": sheet_id,
                        "dimension": "COLUMNS",
                        "startIndex": last_data_col,
                        "endIndex": total_col_count
                    }
                }
            })
        
        # Execute batch update if there are changes
        if requests:
            delete_request = {"requests": requests}
            
            sheet_service.spreadsheets().batchUpdate(
                spreadsheetId=spreadsheet_id,
                body=delete_request
            ).execute()
            
            rows_trimmed = total_row_count - last_data_row if last_data_row < total_row_count else 0
            cols_trimmed = total_col_count - last_data_col if last_data_col < total_col_count else 0
            
            print(f"✓ Trimmed {rows_trimmed} blank rows and {cols_trimmed} blank columns from tab '{sheet_name}'.")
            print(f"✓ New dimensions: {last_data_row} rows × {last_data_col} columns")
            
            return last_data_row, last_data_col
        else:
            print(f"✓ No blank rows or columns to trim in tab '{sheet_name}'.")
            return total_row_count, total_col_count
            
    except Exception as e:
        print(f"✗ Error processing tab '{sheet_name}': {e}")
        return total_row_count, total_col_count


def trim_all_tabs(sheet_service, spreadsheet_id):
    """Trim blank rows and columns from ALL tabs in the spreadsheet"""
    
    # Get all sheet names
    spreadsheet = sheet_service.spreadsheets().get(
        spreadsheetId=spreadsheet_id,
        fields="sheets(properties(title))"
    ).execute()
    
    sheet_names = [sheet["properties"]["title"] for sheet in spreadsheet["sheets"]]
    
    print(f"\n{'#'*60}")
    print(f"TRIMMING ALL TABS IN SPREADSHEET")
    print(f"{'#'*60}")
    print(f"Found {len(sheet_names)} tab(s): {', '.join(sheet_names)}")
    
    results = []
    for sheet_name in sheet_names:
        rows, cols = trim_sheet_to_data(sheet_service, spreadsheet_id, sheet_name)
        results.append({
            'Tab Name': sheet_name,
            'Final Rows': rows,
            'Final Columns': cols
        })
    
    # Print summary of all tabs
    print(f"\n{'='*60}")
    print("SUMMARY - ALL TABS")
    print(f"{'='*60}")
    for result in results:
        print(f"  Tab '{result['Tab Name']}': {result['Final Rows']} rows × {result['Final Columns']} columns")
    print(f"{'='*60}")
    
    return results


# Usage - Trim ALL tabs:
trim_all_tabs(sheet_service, ORIGINAL_SPREADSHEET_ID)

# OR trim a single specific tab:
# trim_sheet_to_data(sheet_service, ORIGINAL_SPREADSHEET_ID, "aligned")


############################################################
TRIMMING ALL TABS IN SPREADSHEET
############################################################
Found 16 tab(s): ONGS, Argentina , Homicidios, Drogas Sinteticas, Pandillas, Tennesse media, Migration (US civil society and instutional), Ambiente, Narcotrafico, Esequibo,  Venezuela, Elites y Crimen, Género, Paz Colombia, US terrorism event - Civil, US terrorism event - Govt

TRIMMING TAB: 'ONGS'
Current dimensions: 213 rows × 10 columns
✓ No blank rows or columns to trim in tab 'ONGS'.

TRIMMING TAB: 'Argentina '
Current dimensions: 29 rows × 9 columns
✓ No blank rows or columns to trim in tab 'Argentina '.

TRIMMING TAB: 'Homicidios'
Current dimensions: 239 rows × 9 columns
✓ No blank rows or columns to trim in tab 'Homicidios'.

TRIMMING TAB: 'Drogas Sinteticas'
Current dimensions: 323 rows × 12 columns
✓ No blank rows or columns to trim in tab 'Drogas Sinteticas'.

TRIMMING TAB: 'Pandillas'
Current dimensions: 305 rows × 14 co

[{'Tab Name': 'ONGS', 'Final Rows': 213, 'Final Columns': 10},
 {'Tab Name': 'Argentina ', 'Final Rows': 29, 'Final Columns': 9},
 {'Tab Name': 'Homicidios', 'Final Rows': 239, 'Final Columns': 9},
 {'Tab Name': 'Drogas Sinteticas', 'Final Rows': 323, 'Final Columns': 12},
 {'Tab Name': 'Pandillas', 'Final Rows': 305, 'Final Columns': 14},
 {'Tab Name': 'Tennesse media', 'Final Rows': 33, 'Final Columns': 4},
 {'Tab Name': 'Migration (US civil society and instutional)',
  'Final Rows': 34,
  'Final Columns': 3},
 {'Tab Name': 'Ambiente', 'Final Rows': 493, 'Final Columns': 16},
 {'Tab Name': 'Narcotrafico', 'Final Rows': 128, 'Final Columns': 9},
 {'Tab Name': 'Esequibo', 'Final Rows': 442, 'Final Columns': 13},
 {'Tab Name': ' Venezuela', 'Final Rows': 278, 'Final Columns': 13},
 {'Tab Name': 'Elites y Crimen', 'Final Rows': 161, 'Final Columns': 10},
 {'Tab Name': 'Género', 'Final Rows': 433, 'Final Columns': 21},
 {'Tab Name': 'Paz Colombia', 'Final Rows': 401, 'Final Columns': 14},

### Email detection and cleaning

* Scans all columns in each row for email patterns
* Fills email if reference column is blank and there is an email in that row anywhere else (first is selected)
* Replaces email if it contains non-email text in the reference column and there is an email in that row anywhere else (first is selected)
* Keeps valid emails unchanged
* Normalizes emails
* Removes duplicates
* Removes rows with missing emails
* Removes rows that are all blank
* Logs duplicates and summaries

In [ ]:
import re
import pandas as pd

EMAIL_REGEX = r'[\w\.-]+@[\w\.-]+\.\w+'

def is_valid_email(val):
    if not isinstance(val, str):
        return False
    return bool(re.match(f"^{EMAIL_REGEX}$", val.strip()))

def extract_email_from_row_excluding_first(row):
    for val in row.iloc[1:]:  # skip first column
        if isinstance(val, str):
            match = re.search(EMAIL_REGEX, val)
            if match:
                return match.group(0)
    return None

cleaned_dfs = {}  # ← dict instead of list
summary_stats = []
duplicate_log = []

for sheet_name, df in sheets_dict.items():
    df = df.copy()
    original_rows = len(df)
    
    # Normalize column names
    df.columns = df.columns.str.strip().str.lower()
    
    # First column = authoritative email column
    first_col = df.columns[0]
    
    # Remove completely blank rows
    blank_mask = df.isna().all(axis=1)
    blank_removed = df[blank_mask]
    df = df[~blank_mask]
    
    # Resolve emails based on first column
    def resolve_email(row):
        first_val = row[first_col]
        
        if is_valid_email(first_val):
            return first_val.strip()
        
        return extract_email_from_row_excluding_first(row)
    
    # Overwrite first column
    df[first_col] = df.apply(resolve_email, axis=1)
    
    # Unified email column
    df["email"] = df[first_col]
    
    # Normalize
    df["email"] = df["email"].where(df["email"].notna(), None)
    df["email"] = df["email"].astype(str).str.lower().str.strip()
    df.loc[df["email"] == "none", "email"] = None
    
    # Remove rows with no email
    no_email_mask = df["email"].isna()
    no_email_removed = df[no_email_mask]
    df = df[~no_email_mask]
    
    # Remove duplicates
    dup_mask = df.duplicated(subset=["email"], keep="first")
    duplicates = df[dup_mask]
    
    for idx, row in duplicates.iterrows():
        duplicate_log.append({
            "Tab": sheet_name,
            "Row Index Removed": idx,
            "Duplicate Email": row["email"]
        })
    
    df = df[~dup_mask]
    
    # Add tab_theme column
    df["tab_theme"] = sheet_name  # ← new column

    cleaned_dfs[sheet_name] = df  # ← store with tab name as key
    
    # Summary
    summary_stats.append({
        'Tab Name': sheet_name,
        'Original Rows': original_rows,
        'Blank Rows Removed': len(blank_removed),
        'No Email Rows Removed': len(no_email_removed),
        'Duplicate Rows Removed': len(duplicates),
        'Final Rows': len(df),
        'Unique Emails': df['email'].nunique()
    })
    
    print(f"\n{'='*50}")
    print(f"SUMMARY FOR TAB: '{sheet_name}'")
    print(f"{'='*50}")
    print(f"  Original rows:           {original_rows}")
    print(f"  Blank rows removed:      {len(blank_removed)}")
    print(f"  No-email rows removed:   {len(no_email_removed)}")
    print(f"  Duplicate rows removed:  {len(duplicates)}")
    print(f"  Final rows:              {len(df)}")
    print(f"  Unique emails:           {df['email'].nunique()}")
    print(f"{'='*50}")

# Logs
print(f"\n{'='*60}")
print("DUPLICATES REMOVED (DETAIL)")
print(f"{'='*60}")
print(pd.DataFrame(duplicate_log).to_string(index=False))

print(f"\n{'='*60}")
print("OVERALL SUMMARY (All Tabs)")
print(f"{'='*60}")
print(pd.DataFrame(summary_stats).to_string(index=False))
print(f"{'='*60}")


SUMMARY FOR TAB: 'ONGS'
  Original rows:           212
  Blank rows removed:      0
  No-email rows removed:   22
  Duplicate rows removed:  36
  Final rows:              154
  Unique emails:           154

SUMMARY FOR TAB: 'Argentina '
  Original rows:           28
  Blank rows removed:      0
  No-email rows removed:   3
  Duplicate rows removed:  0
  Final rows:              25
  Unique emails:           25

SUMMARY FOR TAB: 'Homicidios'
  Original rows:           238
  Blank rows removed:      0
  No-email rows removed:   28
  Duplicate rows removed:  53
  Final rows:              157
  Unique emails:           157

SUMMARY FOR TAB: 'Drogas Sinteticas'
  Original rows:           322
  Blank rows removed:      0
  No-email rows removed:   1
  Duplicate rows removed:  6
  Final rows:              315
  Unique emails:           315

SUMMARY FOR TAB: 'Pandillas'
  Original rows:           304
  Blank rows removed:      0
  No-email rows removed:   1
  Duplicate rows removed:  64
  Fin

#### Looking at each cleaned DataFrame

In [ ]:
for sheet_name, df in cleaned_dfs.items():
    print(f"\n{'='*60}")
    print(f"DataFrame: '{sheet_name}' (Shape: {df.shape})")
    print(f"{'='*60}")
    print(df.head(10))
    print(f"\nColumns: {df.columns.tolist()}")
    print(f"\nEmail sample: {df['email'].head().tolist()}")


DataFrame: 'ONGS' (Shape: (154, 14))
                            email address          first name last name  \
0                        fip@ideaspaz.org        Juan Carlos     Garzón   
1             julia.sekula@igarape.org.br              Julia     Sekula   
2           comunicaciones@sismamujer.org                                 
3                jasava-video@hotmail.com  Jaime Leon Sanchez             
4                  humanas@humanas.org.co                                 
5      Ana.trujillo@comisiondelaverdad.co                 Ana  Trujillo   
6                  floralcaraz@latfem.org                Flor             
7                 lramirez@dejusticia.org                                 
8  mariana.moragomez@globalinitiative.net             Mariana             
9                    macosta@sentiido.com                                 

           sector   idioma   tema de interes 2 tema de interes 1  \
0  Sociedad Civil  Español   Northern Triangle              MS13   


In [ ]:
for sheet_name, df in cleaned_dfs.items():
    print(f"\n📧 Sheet '{sheet_name}' - Found {len(df)} emails:")
    print(df['email'].tolist())


📧 Sheet 'ONGS' - Found 154 emails:
['fip@ideaspaz.org', 'julia.sekula@igarape.org.br', 'comunicaciones@sismamujer.org', 'jasava-video@hotmail.com', 'humanas@humanas.org.co', 'ana.trujillo@comisiondelaverdad.co', 'floralcaraz@latfem.org', 'lramirez@dejusticia.org', 'mariana.moragomez@globalinitiative.net', 'macosta@sentiido.com', 'gabriela.chacon@oagnds.org', 'icuesta@ideaspaz.org', 'cmendoza@dialogos.org.gt', 'katherine.ronderos@eip.org', 'aolaya@conflictronses.org', 'jtoro@athenalab.org', 'michael.osullivan@maoc.eu', 'f.i.mueller@uva.nl', 'marielos@cicloscap.com', 'dunia@reportarsinmiedo.org', 'lesviasalguero@pdh.org.gt', 'gramsey@wola.org', 'edickinson@crisisgroup.org', 'monicamaria87@hotmail.com', 'jagamos@hotmail.com', 'cmolinares@360-grados.co', 'cristina_rodriguezb@hotmail.com', 'juana.cabezas@indepaz.org.co', 'victorellana2009@hotmail.com', 'leonardo@indepaz.org.co', 'sugeysanta@yahoo.es', 'juanguillermo.sepulveda@gmail.com', 'julare67@yahoo.es', 'lhcardon@usbcali.edu.co', 'viv

In [ ]:
for sheet_name, df in cleaned_dfs.items():
    print(f"\n{'='*60}")
    print(f"Sheet '{sheet_name}' - {len(df)} emails detected")
    print(f"{'='*60}")
    
    cols_to_show = ['email'] + [c for c in df.columns if c not in ['email', 'source_sheet', 'theme', 'tab_theme']][:3]
    print(df[cols_to_show].head(20))


Sheet 'ONGS' - 154 emails detected
                                     email  \
0                         fip@ideaspaz.org   
1              julia.sekula@igarape.org.br   
2            comunicaciones@sismamujer.org   
3                 jasava-video@hotmail.com   
4                   humanas@humanas.org.co   
5       ana.trujillo@comisiondelaverdad.co   
6                   floralcaraz@latfem.org   
7                  lramirez@dejusticia.org   
8   mariana.moragomez@globalinitiative.net   
9                     macosta@sentiido.com   
10              gabriela.chacon@oagnds.org   
11                    icuesta@ideaspaz.org   
12                cmendoza@dialogos.org.gt   
13              katherine.ronderos@eip.org   
14               aolaya@conflictronses.org   
15                     jtoro@athenalab.org   
16               michael.osullivan@maoc.eu   
17                      f.i.mueller@uva.nl   
18                  marielos@cicloscap.com   
19              dunia@reportarsinmiedo.org  

### Sectors: Explore & Handle

### Unique sectors by Tab

In [ ]:
sector_summary = []

for sheet_name, df in cleaned_dfs.items():
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()
    
    if "sector" in df.columns:
        unique_sectors = (
            df["sector"]
            .dropna()
            .astype(str)
            .str.strip()
            .replace("", pd.NA)
            .dropna()
            .unique()
            .tolist()
        )
        
        if len(unique_sectors) == 0:
            unique_sectors = ["empty"]
    else:
        unique_sectors = ["Does not have"]
    
    sector_summary.append({
        "sheet": sheet_name,  # ← actual tab name instead of sheet_0, sheet_1...
        "unique_sectors": unique_sectors,
        "n_unique": len(unique_sectors)
    })

sector_table = pd.DataFrame(sector_summary)

sector_table

,sheet,unique_sectors,n_unique
0,ONGS,"[Sociedad Civil, Academia, Organismo Internaci...",5
1,Argentina,[Medio de comunicación],1
2,Homicidios,"[Academia, Medios, Gobierno, Think Tank, NGO, ...",6
3,Drogas Sinteticas,"[Medio de Comunicación, Academia, Organismo Or...",14
4,Pandillas,"[Medio de comunicación, Sociedad Civil, Academ...",8
5,Tennesse media,[Does not have],1
6,Migration (US civil society and instutional),[Does not have],1
7,Ambiente,"[Medio de Comunicación, Gobierno, Sociedad Civ...",13
8,Narcotrafico,"[Medio de Comunicación, Sociedad Civil, Consul...",5
9,Esequibo,"[Sector Privado, Otro, Medio de Comunicación, ...",9


In [ ]:
# names of all unique values in the unique_sectors column
all_unique_sectors = set()  
for _, row in sector_table.iterrows():
    all_unique_sectors.update(row["unique_sectors"]) 

print(all_unique_sectors)   

{'NGO', 'Ex gobierno', 'Medio de Comunicación', 'gobierno', 'Estudiante', 'Medio de comunicaicón', 'Sector Privado', 'Civil Society', 'Centro de pensamiento', 'Think tank', 'No aplica', 'Periodistas', 'N/A', 'Medio de comunicación', 'Sector privado', 'Otro', 'Análisis de riesgo y/o seguridad privada', 'Consultoría', 'Embajadas', 'No Gubernamental (ONG)', 'Medios', 'Desconocido', 'Organismo internacional', 'Does not have', 'Gobierno', 'Think Tank', 'Privado', 'No especificado', 'medio de comunicación', 'Organismo Organismo Internacional', 'Academia', 'Sociedad civil', 'Sociedad Civil', 'Organización internacional', 'ONG', 'Organismo Internacional'}


### Map and clean sectors

In [ ]:
sector_mapping = {
    
# 1. Government
    'gobierno': 'Government',
    'Gobierno': 'Government',
    'Ex gobierno': 'Government',
    'Embajadas': 'Government',
        
# 2 Media
    'medio de comunicación': 'Media',
    'Medio de comunicaicón': 'Media',
    'Medio de Comunicación': 'Media',
    'Medios': 'Media',
    'Periodistas': 'Media',

# 3. Civil Society / NGO / Academia 
    'Sociedad civil': 'Civil Society, NGO & Academia',
    'Sociedad Civil': 'Civil Society, NGO & Academia',
    'Civil Society': 'Civil Society, NGO & Academia',
    'NGO': 'Civil Society, NGO & Academia',
    'No Gubernamental (ONG)': 'Civil Society, NGO & Academia',
    'ONG': 'Civil Society, NGO & Academia',
    'Academia': 'Civil Society, NGO & Academia',
    'Estudiante': 'Civil Society, NGO & Academia',

# 4. Private Sector / Consulting / Think Tanks / Analysis 

    'Sector Privado': 'Private Sector, Consulting & Analysis',
    'Sector privado': 'Private Sector, Consulting & Analysis',
    'Privado': 'Private Sector, Consulting & Analysis',
    'Consultoría': 'Private Sector, Consulting & Analysis',
    'Centro de pensamiento': 'Private Sector, Consulting & Analysis',
    'Think Tank': 'Private Sector, Consulting & Analysis',
    'Think tank': 'Private Sector, Consulting & Analysis',
    'Análisis de riesgo y/o seguridad privada': 'Private Sector, Consulting & Analysis',

# 5. International Organizations
    'Organización internacional': 'International Organizations',
    'Organismo Internacional': 'International Organizations',
    'Organismo Organismo Internacional': 'International Organizations',
    'Organismo internacional': 'International Organizations',

# 6. Unknown 
    'No aplica': 'Unknown',
    'No especificado': 'Unknown',
    'Does not have': 'Unknown',
    'N/A': 'Unknown',
    'Desconocido': 'Unknown',
    'Otro': 'Unknown'
}

In [ ]:
for sheet_name, df in cleaned_dfs.items():
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()
    
    if "sector" not in df.columns:
        print(f"⚠️  '{sheet_name}' has no 'sector' column. Columns found: {df.columns.tolist()}")
        cleaned_dfs[sheet_name] = df
        continue

    df["sector_clean"] = df["sector"].map(sector_mapping).fillna("Unknown")
    cleaned_dfs[sheet_name] = df

# Display results from first valid df that has sector_clean
for sheet_name, df in cleaned_dfs.items():
    if "sector_clean" in df.columns:
        print(f"Sample from '{sheet_name}':")
        print(df[["sector", "sector_clean"]].drop_duplicates().sort_values("sector"))
        break

⚠️  'Tennesse media' has no 'sector' column. Columns found: ['name', 'media outlet', 'email', 'language', 'source_sheet', 'theme', 'tab_theme']
⚠️  'Migration (US civil society and instutional)' has no 'sector' column. Columns found: ['name', 'organization/position', 'email', 'source_sheet', 'theme', 'tab_theme']


⚠️  ' Venezuela' has no 'sector' column. Columns found: ['email address', 'first name', 'last name', 'media', 'sector x', 'country', 'language', 'tags', 'tema de interes 1', 'tema de interes 2', 'tema de interes 3', 'segmentación medios', 'segmentación gob, soc civ', 'source_sheet', 'theme', 'email', 'tab_theme']
Sample from 'ONGS':
                      sector                           sector_clean
25                  Academia          Civil Society, NGO & Academia
179                      NGO          Civil Society, NGO & Academia
68   Organismo Internacional            International Organizations
0             Sociedad Civil          Civil Society, NGO & Academia
173               Think Tank  Private Sector, Consulting & Analysis


In [ ]:
for sheet_name, df in cleaned_dfs.items():
    print(f"\n{'='*60}")
    print(f"DataFrame '{sheet_name}' (Shape: {df.shape})")
    print(f"{'='*60}")
    print(df.head(10))
    print(f"\nColumns: {df.columns.tolist()}")
    print(f"\nEmail sample: {df['email'].head().tolist()}")


DataFrame 'ONGS' (Shape: (154, 15))
                            email address          first name last name  \
0                        fip@ideaspaz.org        Juan Carlos     Garzón   
1             julia.sekula@igarape.org.br              Julia     Sekula   
2           comunicaciones@sismamujer.org                                 
3                jasava-video@hotmail.com  Jaime Leon Sanchez             
4                  humanas@humanas.org.co                                 
5      Ana.trujillo@comisiondelaverdad.co                 Ana  Trujillo   
6                  floralcaraz@latfem.org                Flor             
7                 lramirez@dejusticia.org                                 
8  mariana.moragomez@globalinitiative.net             Mariana             
9                    macosta@sentiido.com                                 

           sector   idioma   tema de interes 2 tema de interes 1  \
0  Sociedad Civil  Español   Northern Triangle              MS13   
1

## Output by Sectors

In [ ]:
import os

os.makedirs("by_sector", exist_ok=True)

def dedupe_columns(df):
    cols = pd.Series(df.columns)
    for dup in cols[cols.duplicated()].unique():
        mask = cols == dup
        cols[mask] = [dup if i == 0 else f"{dup}_{i}" for i, _ in enumerate(mask[mask].index)]
    df.columns = cols
    return df

master_df_pre = pd.concat(
    [dedupe_columns(df) for df in cleaned_dfs.values()],
    ignore_index=True,
    sort=False
)

with pd.ExcelWriter("by_sector_clean.xlsx", engine="openpyxl") as writer:
    for sector, df in master_df_pre.groupby("sector_clean"):
        safe_name = sector.strip()[:31]
        df.to_excel(writer, sheet_name=safe_name, index=False)
        print(f"✅ Saved sheet: {safe_name}  ({len(df)} rows)")

✅ Saved sheet: Civil Society, NGO & Academia  (835 rows)
✅ Saved sheet: Government  (175 rows)
✅ Saved sheet: International Organizations  (84 rows)
✅ Saved sheet: Media  (692 rows)
✅ Saved sheet: Private Sector, Consulting & An  (116 rows)
✅ Saved sheet: Unknown  (828 rows)


### Combine sheets

✅  'ONGS' columns OK
✅  'Argentina ' columns OK
✅  'Homicidios' columns OK
✅  'Drogas Sinteticas' columns OK
✅  'Pandillas' columns OK
✅  'Tennesse media' columns OK
✅  'Migration (US civil society and instutional)' columns OK
✅  'Ambiente' columns OK
⚠️  'Narcotrafico' has duplicate columns: ['interés']
✅  'Esequibo' columns OK
✅  ' Venezuela' columns OK
✅  'Elites y Crimen' columns OK
✅  'Género' columns OK
✅  'Paz Colombia' columns OK
✅  'US terrorism event - Civil' columns OK
✅  'US terrorism event - Govt' columns OK


🔧 Fixed duplicate columns in 'Narcotrafico'

Total rows after concat: 3051
Columns: ['email address', 'first name', 'last name', 'sector', 'idioma', 'tema de interes 2', 'tema de interes 1', 'tema de interes 3', 'organización', 'extra_col_9', 'source_sheet', 'theme', 'email', 'tab_theme', 'sector_clean', 'tema de interes', 'extra_col_7', 'extra_col_8', 'correo', 'nombre', 'apellido', 'interés 1', 'interés 2', 'país', 'investigación', 'tema de interes 4', 'ubicación', 'correo evento', 'interés 3', 'tema de interés4', 'extra_col_11', 'extra_col_12', 'extra_col_13', 'name', 'media outlet', 'language', 'organization/position', 'investigación o base de datos', 'extra_col_14', 'extra_col_15', 'media', '', 'interés', 'interés_1', 'country', 'tags', 'civ soc', 'medios', 'sector x', 'segmentación medios', 'segmentación gob, soc civ', 'first', 'tema de interés 1', 'tema de interés 2', 'tipo corrupción', 'tema de interes 5', 'tema de interes 6', 'extra_col_16', 'extra_col_17', 'extra_col_18', 'ex

## Delivery

In [ ]:
master_df.to_csv("mailchimp-clean-upload.csv", index=False)
print("✅ Saved as mailchimp-clean-upload.csv")


✅ Saved as mailchimp-clean-upload.csv


# End

In [ ]:
end_time = time.time()
end_readable = time.ctime(end_time)
execution_time = (end_time - start_time)/60


print(f"Start: {formatted_start_time}")
print(f"End: {end_readable}")
print(f"Execution time: {execution_time:.2f} minutes")

print("="*80)

Start: Tue Apr  7 07:28:56 2026
End: Tue Apr  7 07:29:38 2026
Execution time: 0.70 minutes
